In [1]:
from unet_model import *
# from sleap.nn.model import Model
from sleap.nn.architectures import UNet
from sleap.nn.heads import MaskHead

In [8]:
mask_model = MaskModel(backbone=UNet(), heads=[MaskHead()]).make_model((256, 256, 3))

In [9]:
mask_model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input (InputLayer)             [(None, 256, 256, 1  0           []                               
                                )]                                                                
                                                                                                  
 stack0_enc0_conv0 (Conv2D)     (None, 256, 256, 64  640         ['input[0][0]']                  
                                )                                                                 
                                                                                                  
 stack0_enc0_act0_relu (Activat  (None, 256, 256, 64  0          ['stack0_enc0_conv0[0][0]']      
 ion)                           )                                                           

## mask encoding

In [10]:
# Bullet order based on angular position around centroid
num_identities = 17
angles = np.linspace(0, 2*np.pi, num_identities, endpoint=False)
identity_embeddings = np.stack([np.cos(angles), np.sin(angles)], axis=1)

## temporal UNet

Output Mask Structure (instance segmentation with displacement fields):

Channel 0: Semantic mask (1 if any bulb exists at pixel)

Channel 1: X-offset map (signed distance to nearest bulb's x-coordinate)

Channel 2: Y-offset map (signed distance to nearest bulb's y-coordinate)

Channel 3-N: One-hot identity channels (learned positional embeddings)

In [14]:
from tensorflow.keras.layers import Conv3D, MaxPool3D, UpSampling3D, concatenate, ConvLSTM2D

# TODO: leverage original UNet model from sleap

def TemporalUNet(input_shape=(5, 170, 174, 1), num_identities=17):
    inputs = Input(input_shape)
    
    # Encoder
    c1 = Conv3D(32, (3,3,3), activation='relu', padding='same')(inputs)
    p1 = MaxPool3D((1,2,2))(c1)
    
    c2 = Conv3D(64, (3,3,3), activation='relu', padding='same')(p1)
    p2 = MaxPool3D((1,2,2))(c2)
    
    # Temporal Bottleneck
    c3 = Conv3D(128, (3,3,3), activation='relu', padding='same')(p2)
    
    # Add to bottleneck
    lstm = ConvLSTM2D(128, (3,3), padding='same')(c3)
    
    # Decoder
    u4 = UpSampling3D((1,2,2))(c3)
    c4 = Conv3D(64, (3,3,3), activation='relu', padding='same')(u4)
    m4 = concatenate([c2, c4])
    
    u5 = UpSampling3D((1,2,2))(m4)
    c5 = Conv3D(32, (3,3,3), activation='relu', padding='same')(u5)
    m5 = concatenate([c1, c5])
    
    # Output Heads
    semantic = Conv3D(1, (1,1,1), activation='sigmoid', name='semantic')(m5)
    offsets = Conv3D(2, (1,1,1), activation='tanh', name='offsets')(m5)
    identities = Conv3D(num_identities, (1,1,1), activation='softmax', name='identities')(m5)
    
    return Model(inputs, [semantic, offsets, identities])

## loss function

In [15]:
def temporal_loss(y_true, y_pred):
    # Encourage smooth identity transitions
    return tf.reduce_mean(
        tf.abs(y_pred[:,1:,:,:] - y_pred[:,:-1,:,:])
    )

In [16]:
def composite_loss(y_true, y_pred):
    sem_true, off_true, id_true = y_true
    sem_pred, off_pred, id_pred = y_pred
    
    # Semantic loss (BCE)
    sem_loss = tf.keras.losses.binary_crossentropy(sem_true, sem_pred)
    
    # Offset loss (MSE only on bulb regions)
    mask = tf.cast(sem_true > 0.5, tf.float32)
    off_loss = tf.reduce_mean(tf.square(off_true - off_pred) * mask)
    
    # Identity loss (Focal loss for class imbalance)
    id_loss = tf.keras.losses.categorical_focal_crossentropy(id_true, id_pred)
    
    return 0.5*sem_loss + 1.0*off_loss + 0.2*id_loss

# TODO: check the coefficients of the losses? 

In [ ]:
# how much training data is needed? i.e. how many jellyfish are needed for the model to be able to generalize?
# decode point ID from output? 


## training

Training Tricks:

- Curriculum Learning:

    - Stage 1: Train semantic head only

    - Stage 2: Add offset prediction

    - Stage 3: Enable identity channels

- Data Augmentation:

    - Temporal warping of bulb trajectories

    - Random frame drops (simulate missing data)

- Weight Freezing:
    - Gradually unfreeze encoder layers